In [74]:
from __future__ import annotations

import argparse
import html
import json
import random
import re
import time
import os
import unicodedata
import zlib
from collections import Counter
from dataclasses import dataclass, field

In [75]:
# --------------------------------------------------------------------------
# Stage 1: REPAIR — character-level fixes. These change text, never drop rows.
# --------------------------------------------------------------------------

# PDF/OCR ligatures. Deleting these (as an emoji regex would) silently corrupts
# words: "ﬁnanzas" -> "nanzas". They must be expanded, not removed.

In [76]:
LIGATURES = {
    "ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl",
    "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st", "ﬆ": "st",
}

# Layout glyphs from scanned classifieds / forms. Carry no linguistic content.
LAYOUT_GLYPHS = "■□▪▫●○◆◇➢➣❏❐✔✓✗★☆¶†‡‰•·"

QUOTE_MAP = {
    "’": "'", "‘": "'", "‚": "'", "′": "'",
    "“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
    "‹": "'", "›": "'",
}

DASH_MAP = {"–": "-", "—": "-", "−": "-", "‐": "-", "‑": "-"}

RE_SOFT_HYPHEN = re.compile("­")
RE_DOT_LEADER = re.compile(r"[.․…]{3,}")
RE_UNDERSCORE_RULE = re.compile(r"_{2,}")
RE_LAYOUT = re.compile(f"[{re.escape(LAYOUT_GLYPHS)}]+")
RE_REPEATED_PUNCT = re.compile(r"([^\w\s])\1{2,}")
# Column-wrap hyphenation: "ve- gnius" -> "vegnius". Only fires when the tail
# starts lowercase, which keeps real compounds ("Derungs-Tschuor") intact.
RE_WRAP_HYPHEN = re.compile(r"([^\W\d_])-\s+([a-zäöüàèéìòùâêîôû])")
RE_WS = re.compile(r"\s+")
RE_CONTROL = re.compile(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f​-‏﻿]")

# --- Markup (HTML) ---------------------------------------------------------
# Only names that are real HTML elements are stripped. The corpus contains
# Romansh words in angle brackets (<Strom>, <energia>) used like quotation
# marks; a generic "<[^>]+>" regex would delete them. Those survivors are
# counted in the report as "kept as text", so the decision stays visible.
HTML_TAG_NAMES = {
    "a", "abbr", "b", "big", "blockquote", "body", "br", "center", "div", "em",
    "font", "h1", "h2", "h3", "h4", "h5", "h6", "head", "hr", "html", "i", "img",
    "li", "link", "meta", "ol", "p", "pre", "script", "small", "span", "strong",
    "style", "sub", "sup", "table", "tbody", "td", "th", "thead", "title", "tr",
    "u", "ul",
}
_TAG_ALT = "|".join(sorted(HTML_TAG_NAMES, key=len, reverse=True))
# Attributes must look like name=value. This keeps "<a casa>" (Romansh "a" =
# "to/at") as text, while "<a href='x'>", "<br/>", "</p>" are matched.
_ATTR = r"""\s+[A-Za-z_:][-\w:.]*\s*=\s*(?:"[^"]*"|'[^']*'|[^\s"'<>]+)"""
RE_HTML_TAG = re.compile(rf"</?(?:{_TAG_ALT})(?:{_ATTR})*\s*/?>", re.IGNORECASE)
RE_HTML_BLOCK = re.compile(r"<(script|style)\b[^>]*>.*?</\1\s*>", re.IGNORECASE | re.DOTALL)
RE_HTML_ENTITY = re.compile(r"&(?:#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31});")
# Anything still in angle brackets after repair: reported, never removed.
RE_BRACKETED = re.compile(r"<[^<>\n]{1,60}>")


def _translate(text: str, mapping: dict[str, str]) -> str:
    return text.translate(str.maketrans(mapping))


def _decode_entities(text: str) -> str:
    # Decode only well-formed entities (&amp; &nbsp; &#252;). Unknown names
    # are left untouched by html.unescape, so nothing is guessed.
    return RE_HTML_ENTITY.sub(lambda m: html.unescape(m.group(0)), text)


def _strip_html(text: str) -> str:
    # <script>/<style> blocks go with their contents (code, not text); other
    # tags are replaced by a space. Limitation: a tag *inside* a word
    # ("Gri<i>schun</i>") splits it — rare in newspaper text.
    return RE_HTML_TAG.sub(" ", RE_HTML_BLOCK.sub(" ", text))


# Each repair rule: (name, fn). Order matters:
#   - markup first: it is a layer *on top of* the text, and what it produces
#     (&nbsp; -> \xa0, &shy; -> soft hyphen, &#64257; -> ﬁ) is then cleaned
#     by the rules below;
#   - NFC before any character-level matching;
#   - whitespace collapse last, to clean up the spaces other rules leave.
REPAIR_RULES: list[tuple[str, object]] = [
    ("decode_html_entities", _decode_entities),
    ("strip_html_tags", _strip_html),
    ("nfc_normalize", lambda t: unicodedata.normalize("NFC", t)),
    ("strip_control", lambda t: RE_CONTROL.sub("", t)),
    ("strip_soft_hyphen", lambda t: RE_SOFT_HYPHEN.sub("", t)),
    ("expand_ligatures", lambda t: _translate(t, LIGATURES)),
    ("normalize_quotes", lambda t: _translate(t, QUOTE_MAP)),
    ("normalize_dashes", lambda t: _translate(t, DASH_MAP)),
    ("strip_dot_leaders", lambda t: RE_DOT_LEADER.sub(" ", t)),
    ("strip_underscore_rules", lambda t: RE_UNDERSCORE_RULE.sub(" ", t)),
    ("strip_layout_glyphs", lambda t: RE_LAYOUT.sub(" ", t)),
    ("collapse_repeated_punct", lambda t: RE_REPEATED_PUNCT.sub(r"\1", t)),
    ("join_wrap_hyphens", lambda t: RE_WRAP_HYPHEN.sub(r"\1\2", t)),
    ("collapse_whitespace", lambda t: RE_WS.sub(" ", t).strip()),
]


def repair(text: str) -> tuple[str, list[str]]:
    """Apply every repair rule, recording which ones actually changed the text."""
    applied: list[str] = []
    for name, fn in REPAIR_RULES:
        new = fn(text)
        if new != text:
            applied.append(name)
            text = new
    return text, applied


In [77]:
# --------------------------------------------------------------------------
# Stage 2: FLAG — per-document quality judgements.
# --------------------------------------------------------------------------

In [78]:

@dataclass
class Thresholds:
    min_tokens: int = 10          # below this a "document" is a caption or a header
    max_tokens: int = 2500        # above this it is probably concatenated pages
    min_letter_ratio: float = 0.55  # letters / non-space chars
    max_digit_ratio: float = 0.30   # digit-dense -> results tables, price lists
    min_mean_token_len: float = 2.0  # guards against " . . . . " survivors
    max_ocr_suspicious: float = 0.10  # share of word tokens that look OCR-garbled
    min_reprint_jaccard: float = 0.80  # MinHash similarity for "edited reprint"
    # Single letters ("L L s h n Fi u i r a") = text pulled out of a rotated or
    # letter-spaced layout one character at a time. Normal Romansh has a few
    # ("e", "a"), garbled extraction has dozens.
    max_single_letter_ratio: float = 0.25
    # language ID
    # Real foreign documents in lid_check.txt scored >= 0.98; the false ones
    # (Romansh read as Lombard / Franco-Provençal / Venetian) scored 0.51-0.82.
    lid_doc_min_score: float = 0.90    # whole-document prediction confidence
    lid_zxx_min_score: float = 0.50    # "no linguistic content" -> garbled text
    lid_seg_min_tokens: int = 6        # shorter sentences are merged: LID on 3 words is a guess
    lid_seg_max_tokens: int = 40       # longer runs are cut: ads often have no full stops
    lid_seg_min_score: float = 0.70    # stricter than doc level, segments are short
    lid_min_foreign_tokens: int = 10   # foreign tokens needed to call a doc "mixed"


def letter_ratio(text: str) -> float:
    non_space = sum(1 for c in text if not c.isspace())
    if not non_space:
        return 0.0
    return sum(1 for c in text if c.isalpha()) / non_space


def digit_ratio(text: str) -> float:
    non_space = sum(1 for c in text if not c.isspace())
    if not non_space:
        return 0.0
    return sum(1 for c in text if c.isdigit()) / non_space


def flag_document(text: str, th: Thresholds) -> list[str]:
    """Return the names of every quality rule this document trips."""
    flags: list[str] = []
    if not text.strip():
        return ["empty"]

    tokens = text.split()
    if len(tokens) < th.min_tokens:
        flags.append("too_short")
    if len(tokens) > th.max_tokens:
        flags.append("too_long")
    if letter_ratio(text) < th.min_letter_ratio:
        flags.append("low_letter_ratio")
    if digit_ratio(text) > th.max_digit_ratio:
        flags.append("digit_heavy")
    if tokens and sum(len(t) for t in tokens) / len(tokens) < th.min_mean_token_len:
        flags.append("fragmented_tokens")
    if len(tokens) >= th.min_tokens and single_letter_ratio(tokens) > th.max_single_letter_ratio:
        flags.append("garbled_extraction")
    return flags


def single_letter_ratio(tokens: list[str]) -> float:
    """Share of tokens that are one letter on their own ("L L s h n")."""
    if not tokens:
        return 0.0
    return sum(1 for t in tokens if len(t) == 1 and t.isalpha()) / len(tokens)


# --- OCR damage score ------------------------------------------------------
# Repair fixes errors that follow a *pattern* (ligatures, wrap hyphens). OCR
# also makes *random* errors: a letter read as a digit, a stray capital, a
# symbol. We cannot fix those without a lexicon, but we can count tokens that
# look garbled and score each document by the share of such tokens.
#
# Each pattern is anchored at a token start, so one token counts at most once
# per pattern; the union over patterns gives the number of suspicious tokens.
# Known false positives are designed out and listed next to each pattern.
_L = r"[^\W\d_]"                 # any letter (Unicode-aware)
_LOW = r"[a-zß-öø-ÿ]"            # lowercase, incl. Romansh accented letters
_UP = r"[A-ZÀ-ÖØ-Þ]"
OCR_PATTERNS = {
    # c0mün, vegn1us. Not "km2", "A1", "CH-7000" (no letter on both sides).
    "digit_in_word": rf"{_L}\d+{_L}",
    # geNeral. Not "RhB", "GmbH", "McDonald" (needs 2 lowercase before, 1 after).
    # NOT used: "inner_capital" (geNeral). On the real corpus its top hits were
    # all correct text: names (AlpTransit, PostFinance), Romansh gender-inclusive
    # plurals (scolarAs, delegadAs) and names that lost their hyphen in PDF
    # extraction (WidmerSchlumpf). It caused 7,786 of 9,008 "suspicious" docs.
    # ch|ü, cum£n. "@" left out on purpose: e-mail addresses are real text.
    "symbol_in_word": rf"{_L}[|£¢¤¥§~^`{{}}\\]{_L}",
    # cuuumün. "w" left out on purpose: "www." in URLs.
    "letter_stutter": r"([a-vx-zß-öø-ÿ])\1\1",
}
RE_OCR = {name: re.compile(rf"(?<!\S)\S*?(?:{pat})") for name, pat in OCR_PATTERNS.items()}
# A lowercase token of 5+ letters with no vowel ("tschnt", "grnd") is not a
# Romansh word. "y" counts as a vowel.
RE_NO_VOWEL = re.compile(r"(?<!\S)[b-df-hj-np-tv-xz]{5,}[.,;:!?)\"']*(?!\S)")
RE_WORD_TOKEN = re.compile(r"(?<!\S)\S*?[^\W\d_]")  # tokens containing a letter


def ocr_score(text: str) -> tuple[float, list[str]]:
    """Share of word tokens that look OCR-garbled, plus which patterns fired."""
    n_words = len(RE_WORD_TOKEN.findall(text))
    if not n_words:
        return 0.0, []
    suspicious: set[int] = set()   # token start offsets
    fired: list[str] = []
    for name, rx in [*RE_OCR.items(), ("no_vowel", RE_NO_VOWEL)]:
        starts = {m.start() for m in rx.finditer(text)}
        if starts:
            fired.append(name)
            suspicious |= starts
    return len(suspicious) / n_words, fired

In [79]:
# --------------------------------------------------------------------------
# Stage 3: CORPUS — signals requiring a global view.
# --------------------------------------------------------------------------

In [80]:
RE_NONALNUM = re.compile(r"[^\w\s]", re.UNICODE)
RE_DIGITS = re.compile(r"\d+")


def norm_key(text: str, mask_digits: bool = True) -> str:
    """Case/punctuation-insensitive key. Catches near-identical reprints that
    an exact `set()` check misses — the corpus reports 0 exact duplicates,
    which means it was exact-deduped upstream, not that it has no repeats.

    `mask_digits` is the aggressiveness dial, and it matters: with digits
    masked, two documents differing only in numbers (sports result tables,
    price lists, obituaries differing in postcode) collapse into one key. For
    a template-heavy newspaper corpus that is usually what you want; report
    the dedup rate both ways if you are unsure.
    """
    t = text.casefold()
    if mask_digits:
        t = RE_DIGITS.sub("#", t)
    t = RE_NONALNUM.sub(" ", t)
    return RE_WS.sub(" ", t).strip()


def edge_signature(text: str, n: int = 6) -> tuple[str, str]:
    """First n and last n normalized tokens. Obituary and classified-ad
    templates open *and* close identically ("en tgeua malencurada", "unser
    angebot", "adressa da malencurada"), so a high-frequency edge signature
    identifies a boilerplate *family* rather than a single duplicate.

    n must stay small: at n=8 the obituary family splits apart, because the
    deceased's name falls inside the window and differs row to row.

    Limitation to state in the write-up: this only catches templates anchored
    at a document edge. Boilerplate buried mid-document is not detected, so
    treat the counts below as a lower bound.
    """
    toks = norm_key(text).split()
    if not toks:
        return "", ""
    return " ".join(toks[:n]), " ".join(toks[-n:])


def corpus_pass(texts: list[str], boilerplate_min_family: int = 25,
                sig_tokens: int = 6) -> dict[str, list]:
    """Second pass over repaired texts: duplicate and boilerplate flags."""
    keys = [norm_key(t) for t in texts]
    sigs = [edge_signature(t, sig_tokens) for t in texts]

    key_counts = Counter(keys)
    head_counts = Counter(h for h, _ in sigs if h)
    tail_counts = Counter(t for _, t in sigs if t)

    seen: set[str] = set()
    is_dup, is_boiler, family_size, family_sig = [], [], [], []
    for key, (head, tail) in zip(keys, sigs):
        dup = key in seen and key_counts[key] > 1
        seen.add(key)
        is_dup.append(dup)

        h_size = head_counts.get(head, 0)
        t_size = tail_counts.get(tail, 0)
        if h_size >= t_size:
            size, sig = h_size, f"HEAD: {head}"
        else:
            size, sig = t_size, f"TAIL: {tail}"
        family_size.append(size)
        family_sig.append(sig if size >= boilerplate_min_family else "")
        is_boiler.append(size >= boilerplate_min_family)
    return {
        "near_duplicate": is_dup,
        "recurring_format": is_boiler,
        "lead_family_size": family_size,
        "family_signature": family_sig,
    }


# --- Edited reprints (MinHash + LSH) ----------------------------------------
# norm_key only matches reprints whose words are *identical*. An article
# reprinted with one sentence changed gets a different key. MinHash estimates
# the Jaccard similarity of two documents' 5-word shingle sets:
#
#   shingles("la fabrica da la halla da glatsch") =
#       {"la fabrica da la halla", "fabrica da la halla da", "da la halla da glatsch"}
#   Jaccard(A, B) = |A ∩ B| / |A ∪ B|
#
# Comparing all pairs of 145k documents is 10 billion comparisons. LSH avoids
# that: the 128-number signature is cut into 16 bands of 8; two documents
# become a *candidate pair* only if at least one band is identical. With
# 16x8, a pair at Jaccard 0.8 becomes a candidate ~95% of the time, a pair
# at 0.5 under 7%. Candidates are then verified against min_reprint_jaccard.
_PRIME = 4294967311  # smallest prime > 2**32; keeps the hash arithmetic in uint64


def _shingle_hashes(text: str, k: int) -> list[int]:
    toks = norm_key(text).split()
    if len(toks) < k:
        return []
    # crc32, not hash(): Python's hash() is randomized per run -> not reproducible.
    return list({zlib.crc32(" ".join(toks[i:i + k]).encode("utf-8"))
                 for i in range(len(toks) - k + 1)})


def minhash_signatures(texts: list[str], num_perm: int = 128, k: int = 5,
                       seed: int = 1, min_shingles: int = 10):
    """One row of num_perm minimum hash values per document. Documents with
    fewer than min_shingles shingles (~14 tokens) are marked invalid: their
    Jaccard estimates are too noisy to call anything a reprint."""
    import numpy as np

    rng = np.random.RandomState(seed)
    a = rng.randint(1, 2**31, size=num_perm, dtype=np.int64).astype(np.uint64)
    b = rng.randint(0, 2**31, size=num_perm, dtype=np.int64).astype(np.uint64)
    prime = np.uint64(_PRIME)

    sigs = np.full((len(texts), num_perm), np.iinfo(np.uint32).max, dtype=np.uint32)
    valid = np.zeros(len(texts), dtype=bool)
    for i, text in enumerate(texts):
        hs = _shingle_hashes(text, k)
        if len(hs) < min_shingles:
            continue
        x = np.asarray(hs, dtype=np.uint64)[:, None]        # (n_shingles, 1)
        # a*x < 2**63 for x < 2**32 and a < 2**31, so no overflow.
        sigs[i] = ((x * a + b) % prime).min(axis=0).astype(np.uint32)
        valid[i] = True
    return sigs, valid


def find_edited_reprints(texts: list[str], already_dup: list[bool], th: Thresholds,
                         num_perm: int = 128, bands: int = 16, k: int = 5,
                         seed: int = 1) -> dict[str, list]:
    """Cluster documents whose estimated Jaccard >= th.min_reprint_jaccard.
    The earliest document of each cluster is kept; later ones are flagged as
    edited reprints — unless norm_key already flagged them as near-duplicates,
    so the two counts in the report do not overlap."""
    rows_per_band = num_perm // bands
    sigs, valid = minhash_signatures(texts, num_perm=num_perm, k=k, seed=seed)

    parent = list(range(len(texts)))  # union-find: parent[i] == i means root

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i: int, j: int) -> None:
        ri, rj = find(i), find(j)
        if ri != rj:  # the smaller index becomes the root -> earliest doc is kept
            parent[max(ri, rj)] = min(ri, rj)

    valid_idx = [int(i) for i in valid.nonzero()[0]]
    checked: set[tuple[int, int]] = set()
    for band in range(bands):
        cols = slice(band * rows_per_band, (band + 1) * rows_per_band)
        buckets: dict[bytes, list[int]] = {}
        for i in valid_idx:
            buckets.setdefault(sigs[i, cols].tobytes(), []).append(i)
        for members in buckets.values():
            # Compare every member against the bucket's first document only:
            # O(n) per bucket instead of O(n^2) for big template buckets.
            anchor = members[0]
            for j in members[1:]:
                if (anchor, j) in checked:
                    continue
                checked.add((anchor, j))
                if float((sigs[anchor] == sigs[j]).mean()) >= th.min_reprint_jaccard:
                    union(anchor, j)

    is_reprint, reprint_of, similarity = [], [], []
    for i in range(len(texts)):
        root = find(i)
        if root == i:
            is_reprint.append(False)
            reprint_of.append(-1)
            similarity.append(0.0)
        else:
            is_reprint.append(not already_dup[i])
            reprint_of.append(root)
            similarity.append(round(float((sigs[root] == sigs[i]).mean()), 3))
    return {"edited_reprint": is_reprint, "reprint_of": reprint_of,
            "reprint_similarity": similarity}


# --- Metadata sanity checks -------------------------------------------------
VALID_YEARS = (1997, 2008)          # the "1997_2008" config
RE_YEAR = re.compile(r"\b(1[89]\d{2}|20\d{2})\b")
META_COLUMNS = ("variety", "pub_date", "post_id", "file_name")


def metadata_pass(rows: list[dict], year_range: tuple[int, int] = VALID_YEARS):
    """Per-row metadata issues + a corpus summary (variety x year table).
    Only columns that exist in the data are checked."""
    present = set(rows[0]) if rows else set()
    issues: list[list[str]] = []
    seen_ids: set = set()
    variety_year: Counter = Counter()

    for row in rows:
        iss: list[str] = []
        variety = str(row.get("variety") or "").strip()
        year = None
        if "variety" in present:
            if not variety:
                iss.append("missing_variety")
            elif not variety.startswith("rm"):
                iss.append("unknown_variety")
        if "pub_date" in present:
            m = RE_YEAR.search(str(row.get("pub_date") or ""))
            if not m:
                iss.append("unparseable_date")
            else:
                year = int(m.group(1))
                if not year_range[0] <= year <= year_range[1]:
                    iss.append("date_out_of_range")
        if "post_id" in present:
            pid = row.get("post_id")
            if pid is None or pid == "":
                iss.append("missing_post_id")
            elif pid in seen_ids:
                iss.append("duplicate_post_id")
            else:
                seen_ids.add(pid)
        if "file_name" in present and not row.get("file_name"):
            iss.append("missing_file_name")
        issues.append(iss)
        variety_year[(variety or "?", year if year is not None else "?")] += 1

    summary = {
        "columns_checked": [c for c in META_COLUMNS if c in present],
        "issue_counts": Counter(i for iss in issues for i in iss),
        "unique_post_ids": len(seen_ids),
        "variety_year": variety_year,
        "year_range": year_range,
    }
    return issues, summary


In [81]:
# --------------------------------------------------------------------------
# language identification (GlotLID / lid.176 via fasttext-predict-v2)
# --------------------------------------------------------------------------

In [82]:
ROMANSH_LABELS = {"__label__roh_Latn", "__label__roh", "__label__rm"}

# Foreign = an ALLOW-LIST of languages verified by reading samples (lid_check.py
# on the full corpus). Every other label GlotLID gave was, in the samples,
# Romansh misread as a Romance neighbour: Vallader and Puter as Lombard,
# Sutsilvan as Emilian / Ligurian, plus Occitan, Catalan, Sardinian, Venetian,
# Sicilian, Portuguese. Filtering on "anything not Romansh" flagged 15.9% of
# Puter but 1.0% of Sursilvan documents. Labels outside the lists count as Romansh.
DOC_FOREIGN_LABELS = {
    "__label__deu_Latn", "__label__gsw_Latn", "__label__bar_Latn",  # German, Swiss German
    "__label__eng_Latn", "__label__fra_Latn", "__label__ita_Latn",
    "__label__spa_Latn", "__label__lat_Latn",
    "__label__fur_Latn", "__label__lld_Latn",   # real Friulian / Ladin articles at doc level
}
# Per sentence, Italian is left out: 7 of 8 sampled "Italian" sentences were
# Romansh sentences containing Italian names or titles. Friulian / Ladin were
# not verified at sentence level.
SEG_FOREIGN_LABELS = DOC_FOREIGN_LABELS - {
    "__label__ita_Latn", "__label__fur_Latn", "__label__lld_Latn",
    # Bavarian / Swiss German dialect: 5 of 8 sampled sentences were Romansh
    # (mostly Sutsilvan, Puter) and 2 were garbled text.
    "__label__bar_Latn"}
# GlotLID's "no linguistic content". In the samples: text extracted one letter
# at a time from rotated/letter-spaced layouts. A text problem, not a language.
NO_LANGUAGE_LABEL = "__label__zxx_Latn"


def load_lid(model_path: str):
    def load_lid(model_path: str):
    
        # Auto-download GlotLID model if the file doesn't exist locally
        if not os.path.exists(model_path):
            print(f"[INFO] {model_path} not found locally — downloading from HuggingFace...")
            try:
                from huggingface_hub import hf_hub_download
                model_path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin",
                                             local_dir=".")
                print(f"[INFO] Downloaded to {model_path}")
            except Exception as e:
                raise FileNotFoundError(
                    f"{model_path} not found and auto-download failed: {e}\n"
                    "Install huggingface_hub:  pip install huggingface_hub\n"
                    "Or download manually:     "
                    "from huggingface_hub import hf_hub_download; "
                    "hf_hub_download('cis-lmu/glotlid', 'model_v3.bin', local_dir='.')"
                ) from e
        import fasttext  # provided by fasttext-predict-v2
        return fasttext.load_model(model_path)


def lid_predict(model, text: str) -> tuple[str, float]:
    labels, scores = model.predict(text.replace("\n", " ")[:2000], k=1)
    return (labels[0], float(scores[0])) if labels else ("__label__unk", 0.0)


# --- Per-sentence LID --------------------------------------------------------
# Whole-document LID answers "what language is MOST of this?". A Romansh
# article with a German job ad glued to its end is still "mostly Romansh", so
# the ad survives. Classifying each sentence finds those foreign pieces.

# Sentence boundary: end punctuation, whitespace, then something that can
# start a sentence (capital, digit, opening quote or bracket). Abbreviations
# ("S. Benedetg", "eidg. Fachausweis") cause extra splits; that is harmless,
# because short pieces are merged back together in lid_segments().
RE_SENT_SPLIT = re.compile(r"(?<=[.!?:;])\s+(?=[\"'(\[]?[A-ZÀ-ÖØ-Þ0-9])")


def lid_segments(text: str, min_tokens: int = 6, max_tokens: int = 40) -> list[str]:
    """Split a document into sentence-like segments for LID.

    - Sentences longer than max_tokens are cut into windows. Classified ads
      often have no full stops, so without this an ad glued to an article
      would hide inside one long "sentence".
    - Pieces shorter than min_tokens are merged with the next piece (the
      last one with the previous), because LID on 2-3 words is unreliable.

    Joining the segments with " " gives back the input text exactly, since
    collapse_whitespace already ran.
    """
    pieces: list[list[str]] = []
    for sent in RE_SENT_SPLIT.split(text):
        toks = sent.split()
        for i in range(0, len(toks), max_tokens):
            pieces.append(toks[i:i + max_tokens])

    segs: list[list[str]] = []
    buf: list[str] = []
    for toks in pieces:
        buf.extend(toks)
        if len(buf) >= min_tokens:
            segs.append(buf)
            buf = []
    if buf:
        if segs:
            segs[-1].extend(buf)
        else:
            segs.append(buf)
    return [" ".join(s) for s in segs]


def sentence_lid(model, text: str, th: Thresholds) -> dict:
    """Classify every segment. A segment counts as foreign when it is long
    enough, confidently predicted, and its label is in SEG_FOREIGN_LABELS."""
    segs = lid_segments(text, th.lid_seg_min_tokens, th.lid_seg_max_tokens)
    kept: list[str] = []
    foreign: list[tuple[str, float, str]] = []
    ignored: list[str] = []   # confident non-Romansh labels NOT on the allow-list
    n_tokens = n_foreign = 0
    for seg in segs:
        label, score = lid_predict(model, seg)
        n = len(seg.split())
        n_tokens += n
        confident = n >= th.lid_seg_min_tokens and score >= th.lid_seg_min_score
        if confident and label in SEG_FOREIGN_LABELS:
            n_foreign += n
            foreign.append((label, round(score, 3), seg))
        else:
            kept.append(seg)
            if confident and label not in ROMANSH_LABELS:
                ignored.append(label)
    return {
        "n_segments": len(segs),
        "ignored_labels": ignored,
        "foreign_tokens": n_foreign,
        "foreign_share": n_foreign / n_tokens if n_tokens else 0.0,
        "foreign_segments": foreign,
        # Same text with the foreign segments removed. Offered as an extra
        # column; `text` itself is not changed, so the choice stays downstream.
        "text_romansh_only": " ".join(kept),
    }

In [83]:
# --------------------------------------------------------------------------
# Stage 4: REPORT
# --------------------------------------------------------------------------

In [84]:
def _pct(sorted_vals: list[float], q: float) -> float:
    if not sorted_vals:
        return 0.0
    return sorted_vals[min(len(sorted_vals) - 1, int(q * len(sorted_vals)))]


@dataclass
class Report:
    n_docs: int = 0
    repair_counts: Counter = field(default_factory=Counter)
    flag_counts: Counter = field(default_factory=Counter)
    examples: dict[str, list] = field(default_factory=dict)
    flag_examples: dict[str, list] = field(default_factory=dict)
    families: Counter = field(default_factory=Counter)
    tokens_before: int = 0
    tokens_after: int = 0
    chars_before: int = 0
    chars_after: int = 0
    # new sections
    bracketed_kept: Counter = field(default_factory=Counter)
    docs_with_brackets: int = 0
    ocr_rates: list = field(default_factory=list)
    ocr_pattern_docs: Counter = field(default_factory=Counter)
    reprint_ran: bool = False
    reprint_clusters: int = 0
    reprint_pairs: list = field(default_factory=list)
    meta: dict = field(default_factory=dict)
    lid_ran: bool = False
    lid_sentence_ran: bool = False
    lid_doc_labels: Counter = field(default_factory=Counter)
    lid_segments: int = 0
    lid_foreign_segments: int = 0
    lid_foreign_labels: Counter = field(default_factory=Counter)
    lid_ignored_segments: Counter = field(default_factory=Counter)
    lid_mixed_majority: int = 0       # mixed docs where foreign tokens >= 50%
    lid_mixed_examples: list = field(default_factory=list)

    def summary_json(self) -> dict:
        rates = sorted(self.ocr_rates)
        meta = dict(self.meta)
        if meta:
            meta["issue_counts"] = dict(meta["issue_counts"])
            meta["variety_year"] = {f"{v}|{y}": c for (v, y), c in meta["variety_year"].items()}
        return {
            "markup": {"docs_entities_decoded": self.repair_counts["decode_html_entities"],
                       "docs_html_stripped": self.repair_counts["strip_html_tags"],
                       "docs_with_bracketed_text_kept": self.docs_with_brackets},
            "ocr": {"mean": sum(rates) / max(len(rates), 1), "p50": _pct(rates, .5),
                    "p90": _pct(rates, .9), "p99": _pct(rates, .99),
                    "docs_any_suspicious": sum(r > 0 for r in rates),
                    "pattern_docs": dict(self.ocr_pattern_docs)},
            "reprints": {"ran": self.reprint_ran, "clusters": self.reprint_clusters,
                         "docs_flagged": self.flag_counts["edited_reprint"]},
            "metadata": meta,
            "lid": {"ran": self.lid_ran, "sentence_ran": self.lid_sentence_ran,
                    "doc_labels": dict(self.lid_doc_labels.most_common(20)),
                    "segments": self.lid_segments,
                    "foreign_segments": self.lid_foreign_segments,
                    "foreign_segment_labels": dict(self.lid_foreign_labels.most_common(20)),
                    "ignored_label_segments": dict(self.lid_ignored_segments.most_common(20)),
                    "docs_no_language": self.flag_counts["lid_no_language"],
                    "docs_mixed": self.flag_counts["lid_mixed_language"],
                    "docs_mixed_majority_foreign": self.lid_mixed_majority},
        }

    def to_markdown(self, th: Thresholds, keep_strict: int, keep_lenient: int) -> str:
        n = max(self.n_docs, 1)
        lines = ["# Cleaning report: ZurichNLP/quotidiana (1997_2008)", ""]
        lines += [
            f"- Documents in: **{self.n_docs:,}**",
            f"- Kept (strict): **{keep_strict:,}** ({keep_strict / n:.2%})",
            f"- Kept (lenient): **{keep_lenient:,}** ({keep_lenient / n:.2%})",
            f"- Characters: {self.chars_before:,} -> {self.chars_after:,} "
            f"({(self.chars_after - self.chars_before) / max(self.chars_before, 1):+.2%})",
            f"- Whitespace tokens: {self.tokens_before:,} -> {self.tokens_after:,} "
            f"({(self.tokens_after - self.tokens_before) / max(self.tokens_before, 1):+.2%})",
            "",
            "## Stage 1 — repair rules (documents changed)",
            "",
            "| rule | docs changed | share |",
            "| --- | ---: | ---: |",
        ]
        for name, _ in REPAIR_RULES:
            c = self.repair_counts[name]
            lines.append(f"| `{name}` | {c:,} | {c / n:.2%} |")

        lines += ["", "## Stage 2/3 — quality flags (documents flagged)", "",
                  "| flag | docs | share |", "| --- | ---: | ---: |"]
        for name, c in self.flag_counts.most_common():
            lines.append(f"| `{name}` | {c:,} | {c / n:.2%} |")

        lines += ["", "## Thresholds", "", "```", json.dumps(th.__dict__, indent=2), "```"]

        # --- markup
        lines += ["", "## Markup (HTML)", "",
                  f"- Documents with HTML entities decoded: "
                  f"**{self.repair_counts['decode_html_entities']:,}**",
                  f"- Documents with real HTML tags stripped: "
                  f"**{self.repair_counts['strip_html_tags']:,}**",
                  f"- Documents with angle-bracketed text **kept** (not HTML): "
                  f"**{self.docs_with_brackets:,}**"]
        if self.bracketed_kept:
            lines += ["", "| kept as text | occurrences |", "| --- | ---: |"]
            for tok, c in self.bracketed_kept.most_common(15):
                lines.append(f"| `{tok}` | {c:,} |")

        # --- OCR
        rates = sorted(self.ocr_rates)
        if rates:
            lines += ["", "## OCR damage score", "",
                      "Share of word tokens per document matching a garbling pattern. "
                      f"Flag `ocr_damaged` above {th.max_ocr_suspicious:.0%} "
                      f"(documents with >= {th.min_tokens} tokens).", "",
                      "| statistic | value |", "| --- | ---: |",
                      f"| mean | {sum(rates) / len(rates):.2%} |",
                      f"| median | {_pct(rates, .5):.2%} |",
                      f"| 90th percentile | {_pct(rates, .9):.2%} |",
                      f"| 99th percentile | {_pct(rates, .99):.2%} |",
                      f"| max | {rates[-1]:.2%} |",
                      f"| docs with >= 1 suspicious token | {sum(r > 0 for r in rates):,} |",
                      "", "| pattern | docs with >= 1 match |", "| --- | ---: |"]
            for name in [*OCR_PATTERNS, "no_vowel"]:
                lines.append(f"| `{name}` | {self.ocr_pattern_docs[name]:,} |")

        # --- reprints
        lines += ["", "## Edited reprints (MinHash)", ""]
        if not self.reprint_ran:
            lines.append("Not run (`--no-minhash`).")
        else:
            lines += [f"- 5-word shingles, 128 permutations, LSH 16 bands x 8 rows, "
                      f"threshold Jaccard >= {th.min_reprint_jaccard}",
                      f"- Clusters with an edited reprint: **{self.reprint_clusters:,}**",
                      f"- Documents flagged `edited_reprint`: "
                      f"**{self.flag_counts['edited_reprint']:,}** "
                      "(excludes docs already flagged `near_duplicate`)"]
            for sim, first, later in self.reprint_pairs:
                lines += ["", f"**estimated Jaccard {sim:.2f}**",
                          f"- kept:    `{first[:220]}`", f"- flagged: `{later[:220]}`"]

        # --- language ID
        lines += ["", "## Language identification (GlotLID)", ""]
        if not self.lid_ran:
            lines.append("Not run (no `--lid-model`).")
        else:
            lines += ["Foreign means a label on an allow-list of languages checked by "
                      "reading samples. Other labels GlotLID gives here (Lombard, Occitan, "
                      "Emilian, ...) were Romansh misread as a Romance neighbour, so they "
                      "count as Romansh.", "",
                      "### Whole document", "",
                      f"- `lid_not_romansh`: label in "
                      f"`{', '.join(sorted(l.replace('__label__', '') for l in DOC_FOREIGN_LABELS))}` "
                      f"with confidence >= {th.lid_doc_min_score}, documents with >= "
                      f"{th.min_tokens} tokens: **{self.flag_counts['lid_not_romansh']:,}**",
                      f"- `lid_no_language`: label `zxx` (no linguistic content) with "
                      f"confidence >= {th.lid_zxx_min_score}: "
                      f"**{self.flag_counts['lid_no_language']:,}**", "",
                      "| predicted label | docs |", "| --- | ---: |"]
            for label, c in self.lid_doc_labels.most_common(10):
                lines.append(f"| `{label}` | {c:,} |")
            lines += ["", "### Per sentence", ""]
            if not self.lid_sentence_ran:
                lines.append("Not run (`--no-sentence-lid`).")
            else:
                lines += [
                    f"Segments: sentences, merged up to >= {th.lid_seg_min_tokens} tokens and "
                    f"cut at {th.lid_seg_max_tokens}. A segment is foreign when its label is in "
                    f"`{', '.join(sorted(l.replace('__label__', '') for l in SEG_FOREIGN_LABELS))}` "
                    f"with confidence >= {th.lid_seg_min_score} (Italian is excluded per "
                    "sentence: sampled 'Italian' sentences were mostly Romansh).", "",
                    f"- Segments classified: **{self.lid_segments:,}**",
                    f"- Foreign segments: **{self.lid_foreign_segments:,}**",
                    f"- Documents flagged `lid_mixed_language` (Romansh overall, >= "
                    f"{th.lid_min_foreign_tokens} foreign tokens): "
                    f"**{self.flag_counts['lid_mixed_language']:,}**",
                    f"- ...of which foreign tokens are the majority: **{self.lid_mixed_majority:,}**",
                    "- Column `text_romansh_only` holds each text with its foreign segments "
                    "removed; `text` is unchanged.", "",
                    "| foreign segment label | segments |", "| --- | ---: |"]
                for label, c in self.lid_foreign_labels.most_common(10):
                    lines.append(f"| `{label}` | {c:,} |")
                if self.lid_ignored_segments:
                    lines += ["", "Confident segments with a label NOT on the allow-list "
                              "(counted as Romansh):", "",
                              "| ignored label | segments |", "| --- | ---: |"]
                    for label, c in self.lid_ignored_segments.most_common(10):
                        lines.append(f"| `{label}` | {c:,} |")
                if self.lid_mixed_examples:
                    lines += ["", "Examples of foreign segments in mixed documents:", ""]
                    for pid, label, score, seg in self.lid_mixed_examples:
                        lines.append(f"- `{pid}` {label} ({score:.2f}): `{seg[:200]}`")

        # --- metadata
        if self.meta:
            m = self.meta
            lines += ["", "## Metadata checks", "",
                      f"- Columns checked: {', '.join(m['columns_checked']) or '(none)'}",
                      f"- Unique post_ids: {m['unique_post_ids']:,}",
                      f"- Expected years: {m['year_range'][0]}-{m['year_range'][1]}", "",
                      "| issue | docs |", "| --- | ---: |"]
            if m["issue_counts"]:
                for issue, c in m["issue_counts"].most_common():
                    lines.append(f"| `{issue}` | {c:,} |")
            else:
                lines.append("| (none) | 0 |")
            vy = m["variety_year"]
            varieties = sorted({v for v, _ in vy})
            years = sorted({y for _, y in vy}, key=lambda y: (isinstance(y, str), y))
            lines += ["", "### Documents per variety and year", "",
                      "| variety | " + " | ".join(map(str, years)) + " | total |",
                      "| --- |" + " ---: |" * (len(years) + 1)]
            for v in varieties:
                cells = [vy.get((v, y), 0) for y in years]
                lines.append(f"| `{v}` | " + " | ".join(f"{c:,}" for c in cells)
                             + f" | {sum(cells):,} |")

        if self.families:
            lines += ["", "## Largest recurring formats (kept, not flagged)", "",
                      "Documents sharing their first or last 6 normalized tokens. Members "
                      "share a format but little text (median 5-gram Jaccard ~0.01 in "
                      "family_check.py), so the column `recurring_format` is information "
                      "for downstream use, not a filter.", "",
                      "| documents | edge signature |", "| ---: | --- |"]
            for sig, c in self.families.most_common(20):
                lines.append(f"| {c:,} | `{sig[:120]}` |")

        lines += ["", "## What each repair rule changed (sampled)", ""]
        for name, pairs in self.examples.items():
            lines += [f"### `{name}`", ""]
            for before, after in pairs:
                lines += [f"- **before:** `{before[:220]}`", f"  **after:**  `{after[:220]}`"]
            lines.append("")

        lines += ["", "## What each filter flagged (sampled)", ""]
        for name, texts in self.flag_examples.items():
            lines += [f"### `{name}`", ""]
            lines += [f"- `{t[:220]}`" for t in texts]
            lines.append("")
        return "\n".join(lines)


In [85]:
# --------------------------------------------------------------------------
# Driver
# --------------------------------------------------------------------------

In [86]:
def _log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def process(rows: list[dict], th: Thresholds, lid_model=None, minhash: bool = True,
            sentence_lid_on: bool = True, n_examples: int = 4,
            seed: int = 0) -> tuple[list[dict], Report]:
    """rows: list of dicts with at least a 'text' key. Returns (rows, report)."""
    rng = random.Random(seed)
    rep = Report(n_docs=len(rows))
    out: list[dict] = []

    # --- Stage 1 + 2: per document
    _log(f"repair + per-document flags on {len(rows):,} docs")
    for row in rows:
        raw = row["text"] or ""
        fixed, applied = repair(raw)
        rep.chars_before += len(raw)
        rep.chars_after += len(fixed)
        rep.tokens_before += len(raw.split())
        rep.tokens_after += len(fixed.split())
        for name in applied:
            rep.repair_counts[name] += 1
            bucket = rep.examples.setdefault(name, [])
            if len(bucket) < n_examples and rng.random() < 0.2:
                bucket.append((raw, fixed))

        brackets = RE_BRACKETED.findall(fixed)
        if brackets:
            rep.docs_with_brackets += 1
            rep.bracketed_kept.update(brackets)

        new = dict(row)
        new["text_raw"] = raw
        new["text"] = fixed
        new["repairs"] = applied
        new["flags"] = flag_document(fixed, th)

        rate, fired = ocr_score(fixed)
        new["ocr_suspicious_rate"] = round(rate, 4)
        rep.ocr_rates.append(rate)
        rep.ocr_pattern_docs.update(fired)
        if rate > th.max_ocr_suspicious and len(fixed.split()) >= th.min_tokens:
            new["flags"].append("ocr_damaged")
        out.append(new)

    # --- Stage 3a: near-duplicates + recurring formats
    _log("near-duplicates + recurring formats")
    # recurring_format is metadata, NOT a flag. family_check.py on the full
    # corpus: members of the largest families (weather reports, job ads,
    # thank-you notices, council reports) share a fixed opening but only
    # ~1% of their 5-word shingles (median Jaccard 0.004-0.017). They repeat a
    # format, not text, so they are kept. Repeated TEXT is caught by
    # near_duplicate and edited_reprint.
    CORPUS_COLUMNS = {"lead_family_size", "family_signature", "recurring_format"}
    corpus = corpus_pass([r["text"] for r in out])
    for i, row in enumerate(out):
        for name, values in corpus.items():
            if name in CORPUS_COLUMNS:
                row[name] = values[i]
            elif values[i]:
                row["flags"].append(name)
    rep.families = Counter(
        r["family_signature"] for r in out if r.get("family_signature")
    )

    # --- Stage 3b: edited reprints
    if minhash:
        _log("edited reprints (MinHash) — the slowest step, a few minutes on the full corpus")
        rep.reprint_ran = True
        rp = find_edited_reprints([r["text"] for r in out], corpus["near_duplicate"], th)
        roots: set[int] = set()
        flagged: list[int] = []
        for i, row in enumerate(out):
            root = rp["reprint_of"][i]
            row["reprint_of"] = str(out[root].get("post_id", root)) if root >= 0 else ""
            row["reprint_similarity"] = rp["reprint_similarity"][i]
            if rp["edited_reprint"][i]:
                row["flags"].append("edited_reprint")
                roots.add(root)
                flagged.append(i)
        rep.reprint_clusters = len(roots)
        for i in rng.sample(flagged, min(n_examples, len(flagged))):
            root = rp["reprint_of"][i]
            rep.reprint_pairs.append((rp["reprint_similarity"][i],
                                      out[root]["text"], out[i]["text"]))

    # --- Stage 3c: language ID
    if lid_model is not None:
        rep.lid_ran = True
        _log("language identification: whole documents")
        for row in out:
            label, score = lid_predict(lid_model, row["text"])
            row["lid_label"], row["lid_score"] = label, round(score, 4)
            rep.lid_doc_labels[label] += 1
            # One-line headers ("En tgeua malencurada:") got labels like Catalan,
            # Polish, Esperanto: LID on 3 words is a guess. Short docs are
            # already flagged too_short, so no LID verdict for them.
            if len(row["text"].split()) < th.min_tokens:
                continue
            if label in DOC_FOREIGN_LABELS and score >= th.lid_doc_min_score:
                row["flags"].append("lid_not_romansh")
            elif label == NO_LANGUAGE_LABEL and score >= th.lid_zxx_min_score:
                row["flags"].append("lid_no_language")

        if sentence_lid_on:
            rep.lid_sentence_ran = True
            _log("language identification: per sentence (slower)")
            mixed: list[tuple] = []
            for k, row in enumerate(out, 1):
                res = sentence_lid(lid_model, row["text"], th)
                row["lid_n_segments"] = res["n_segments"]
                row["lid_foreign_share"] = round(res["foreign_share"], 4)
                row["lid_foreign_labels"] = sorted({lab for lab, _, _ in res["foreign_segments"]})
                row["text_romansh_only"] = res["text_romansh_only"]
                rep.lid_segments += res["n_segments"]
                rep.lid_foreign_segments += len(res["foreign_segments"])
                for lab, _, _ in res["foreign_segments"]:
                    rep.lid_foreign_labels[lab] += 1
                rep.lid_ignored_segments.update(res["ignored_labels"])
                # Docs already flagged lid_not_romansh are foreign as a whole;
                # "mixed" is reserved for documents that pass the doc-level check.
                if ("lid_not_romansh" not in row["flags"]
                        and res["foreign_tokens"] >= th.lid_min_foreign_tokens):
                    row["flags"].append("lid_mixed_language")
                    if res["foreign_share"] >= 0.5:
                        rep.lid_mixed_majority += 1
                    lab, sc, seg = res["foreign_segments"][0]
                    mixed.append((row.get("post_id", k - 1), lab, sc, seg))
                if k % 20000 == 0:
                    _log(f"  {k:,} / {len(out):,} documents")
            rep.lid_mixed_examples = rng.sample(mixed, min(2 * n_examples, len(mixed)))

    # --- Stage 3d: metadata
    _log("metadata checks")
    meta_issues, rep.meta = metadata_pass(rows)
    for row, iss in zip(out, meta_issues):
        row["meta_issues"] = iss
        if iss:
            row["flags"].append("metadata_issue")

    # keep_strict: no flags at all. keep_lenient: tolerate flags that say
    # nothing about the *text* being broken (length/format, templates, metadata).
    # lid_mixed_language is tolerated too: the document is Romansh overall, and
    # text_romansh_only offers the version without the foreign segments.
    LENIENT_OK = {"too_long", "digit_heavy", "metadata_issue",
                  "lid_mixed_language"}
    for row in out:
        flags = set(row["flags"])
        row["keep_strict"] = not flags
        row["keep_lenient"] = not (flags - LENIENT_OK)
        for f in flags:
            rep.flag_counts[f] += 1
            # Sample what each filter actually removes, so the decision to
            # keep or drop the rule can be audited rather than assumed.
            bucket = rep.flag_examples.setdefault(f, [])
            if len(bucket) < n_examples:
                bucket.append(row["text"])

    return out, rep


def audit_sample(rows: list[dict], n_per_flag: int = 20, seed: int = 0) -> list[dict]:
    """Stratified sample for manual precision checking: n documents per flag,
    plus n that survive strictly. Label `verdict` by hand, then precision for
    a rule = share of its sampled docs you agree should have been removed."""
    rng = random.Random(seed)
    by_flag: dict[str, list[dict]] = {}
    for row in rows:
        for f in set(row["flags"]) or {"__kept__"}:
            by_flag.setdefault(f, []).append(row)
    out = []
    for flag, group in sorted(by_flag.items()):
        for row in rng.sample(group, min(n_per_flag, len(group))):
            out.append({
                "flag": flag,
                "post_id": row.get("post_id", ""),
                "variety": row.get("variety", ""),
                "n_tokens": len(row["text"].split()),
                "text": row["text"][:600],
                "verdict": "",  # fill in by hand: correct / incorrect
            })
    return out


def main(argv=None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", default="ZurichNLP/quotidiana")
    ap.add_argument("--config", default="1997_2008")
    ap.add_argument("--split", default="train")
    ap.add_argument("--out", default="./quotidiana_cleaned")
    ap.add_argument("--report", default="cleaning_report.md")
    ap.add_argument("--lid-model", default=None,
                    help="path to GlotLID model_v3.bin or lid.176.ftz")
    ap.add_argument("--no-minhash", action="store_true",
                    help="skip edited-reprint detection (faster debug runs)")
    ap.add_argument("--no-sentence-lid", action="store_true",
                    help="with --lid-model: only classify whole documents")
    ap.add_argument("--limit", type=int, default=None, help="debug on a subset")
    ap.add_argument("--audit", default="audit_sample.csv",
                    help="stratified sample to label by hand")
    ap.add_argument("--audit-n", type=int, default=20)
    # args = ap.parse_args()
    args = ap.parse_args(argv)

    from datasets import Dataset, load_dataset

    ds = load_dataset(args.dataset, args.config)[args.split]
    if args.limit:
        ds = ds.select(range(args.limit))
    rows = [dict(r) for r in ds]

    lid = load_lid(args.lid_model) if args.lid_model else None
    th = Thresholds()
    cleaned, rep = process(rows, th, lid_model=lid, minhash=not args.no_minhash,
                           sentence_lid_on=not args.no_sentence_lid)

    strict = sum(r["keep_strict"] for r in cleaned)
    lenient = sum(r["keep_lenient"] for r in cleaned)

    _log("writing outputs")
    Dataset.from_list(cleaned).save_to_disk(args.out)
    with open(args.report, "w", encoding="utf-8") as fh:
        fh.write(rep.to_markdown(th, strict, lenient))
    with open("cleaning_counts.json", "w", encoding="utf-8") as fh:
        json.dump({"n_docs": rep.n_docs,
                   "repair_counts": dict(rep.repair_counts),
                   "flag_counts": dict(rep.flag_counts),
                   "keep_strict": strict, "keep_lenient": lenient,
                   "thresholds": th.__dict__,
                   **rep.summary_json()}, fh, indent=2, ensure_ascii=False)

    if args.audit:
        import csv
        sample = audit_sample(cleaned, n_per_flag=args.audit_n)
        with open(args.audit, "w", encoding="utf-8", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(sample[0].keys()))
            writer.writeheader()
            writer.writerows(sample)

    print(f"{rep.n_docs:,} in -> strict {strict:,} / lenient {lenient:,}")
    print(f"report: {args.report}   dataset: {args.out}   audit: {args.audit}")


# if __name__ == "__main__":
#     main()
main(["--lid-model", "model_v3.bin"])

[20:33:23] repair + per-document flags on 145,701 docs
[20:38:24] near-duplicates + recurring formats
[20:39:18] edited reprints (MinHash) — the slowest step, a few minutes on the full corpus
[20:40:47] metadata checks
[20:40:48] writing outputs


Saving the dataset (0/1 shards):   0%|          | 0/145701 [00:00<?, ? examples/s]

145,701 in -> strict 145,222 / lenient 145,226
report: cleaning_report.md   dataset: ./quotidiana_cleaned   audit: audit_sample.csv
